In [15]:
# Pretty inline figures + autoreload
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'


In [16]:
 %reload_ext autoreload

In [17]:
# Standard
import os, json, gc, textwrap, glob, pathlib as pl
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# AnnData/scanpy
import scanpy as sc
import anndata as ad
import scipy.sparse as sp

# --- Paths (edit only if your layout changes) ---
LUNA_ROOT_A = "/nfs/team361/mv11/DATASETS/luna/MERFISH_mouse_cortex"
LUNA_ROOT_B = "/nfs/team361/mv11/DATASETS/luna/MERFISH_mouse_cortex/MERFISH_mouse_cortex"
root = LUNA_ROOT_A if os.path.exists(os.path.join(LUNA_ROOT_A, "MERFISH_mouse_cortex_train.csv")) else LUNA_ROOT_B

train_csv = f"{root}/MERFISH_mouse_cortex_train.csv"
test_csv  = f"{root}/MERFISH_mouse_cortex_test.csv"

# Where we will write bronze/silver for VQNiche
DATASETS_ROOT = "/nfs/team361/mv11/DATASETS"
BRONZE_DIR    = f"{DATASETS_ROOT}/bronze"
SILVER_DIR    = f"{DATASETS_ROOT}/silver"

# A dataset name specific to this conversion (becomes subfolder under bronze/silver)
EXPERIMENT_NAME = "merfish_mouse_cortex_from_luna_csv"

# vqniche-reproducibility repo (we’ll call its harmonizer)
REPRO_ROOT = "/nfs/team361/mv11/vqniche-reproducibility"


#### Load CSVs and sanity-check the columns

In [2]:
assert os.path.exists(train_csv) and os.path.exists(test_csv), (train_csv, test_csv)
tr = pd.read_csv(train_csv, index_col=0)
te = pd.read_csv(test_csv,  index_col=0)

NUM_GENES = 254
GENE_COLS = list(tr.columns[:NUM_GENES])
REQ = ["coord_X", "coord_Y", "cell_section", "cell_class"]

for name, df in [("train", tr), ("test", te)]:
    missing = set(REQ) - set(df.columns)
    assert not missing, f"{name} missing {missing}"

assert te.columns[:NUM_GENES].equals(tr.columns[:NUM_GENES]), "Train/Test gene columns mismatch"

print("✓ CSVs OK")
print("Train:", tr.shape, "Test:", te.shape, "| Genes:", len(GENE_COLS))
print("Sections:", tr['cell_section'].nunique(), "(train) /", te['cell_section'].nunique(), "(test)")


✓ CSVs OK
Train: (158379, 262) Test: (118036, 262) | Genes: 254
Sections: 33 (train) / 31 (test)


#### Combine CSVs and build bronze .h5ad per section

In [3]:
# Combine rows; keep the source just in case
tr["split"] = "train"
te["split"] = "test"
df = pd.concat([tr, te], axis=0, ignore_index=False)
del tr, te; gc.collect()

# Prepare bronze output directory
bronze_exp_dir = pl.Path(BRONZE_DIR) / EXPERIMENT_NAME
bronze_exp_dir.mkdir(parents=True, exist_ok=True)

sections = sorted(df["cell_section"].unique().tolist())
print(f"Found {len(sections)} sections.")

def to_int_csr(a: np.ndarray) -> sp.csr_matrix:
    """
    VQNiche's harmonizer insists that counts be integers.
    MERFISH counts *should* be integers; if floats slipped in, we round safely.
    """
    a2 = np.rint(a).astype(np.int32, copy=False)
    return sp.csr_matrix(a2, dtype=np.int32)

# Build one h5ad per section
written = []
for sec in tqdm(sections, desc="Writing bronze h5ad per section"):
    sub = df[df["cell_section"] == sec].copy()

    # Expression matrix (genes only), ensure integers + CSR
    X = to_int_csr(sub[GENE_COLS].to_numpy())

    # .obs — keep useful columns
    obs = sub[["coord_X","coord_Y","cell_section","cell_class","split"]].copy()
    # stable cell ids; add section suffix to avoid clashes across sections
    obs.index = [f"{idx}_{sec}" for idx in sub.index.astype(str)]
    
    # .var — gene symbols as index (harmonizer will attach ensembl IDs)
    var = pd.DataFrame(index=pd.Index(GENE_COLS, name="gene_name"))

    adata = ad.AnnData(X=X, obs=obs, var=var)
    # spatial coordinates (float32, Nx2)
    adata.obsm["spatial"] = sub[["coord_X","coord_Y"]].to_numpy(np.float32)

    # a couple of helpful tags (not required)
    adata.uns["assay"]   = "merfish"
    adata.uns["tissue"]  = "cortex"
    adata.uns["species"] = "mus_musculus"
    adata.uns["section"] = sec

    # filename: keep section in the stem for clarity
    stem = f"merfish_mouse_cortex_{sec}"
    outp = bronze_exp_dir / f"{stem}.h5ad"
    adata.write(outp)
    written.append(str(outp))

len(written), written[:3]


Found 64 sections.


Writing bronze h5ad per section: 100%|██████████████| 64/64 [00:06<00:00, 10.18it/s]


(64,
 ['/nfs/team361/mv11/DATASETS/bronze/merfish_mouse_cortex_from_luna_csv/merfish_mouse_cortex_mouse1_slice1.h5ad',
  '/nfs/team361/mv11/DATASETS/bronze/merfish_mouse_cortex_from_luna_csv/merfish_mouse_cortex_mouse1_slice10.h5ad',
  '/nfs/team361/mv11/DATASETS/bronze/merfish_mouse_cortex_from_luna_csv/merfish_mouse_cortex_mouse1_slice102.h5ad'])

In [8]:
import yaml, pathlib as pl

cfg = {
  "experiment": {
    "name": EXPERIMENT_NAME,
    "description": "MERFISH mouse cortex (LUNA CSV) → silver for VQNiche",
    "study": "LUNA MERFISH mouse cortex",
  },
  "paths": {
    "data_directory_path": None,  # using bronze/silver explicit dirs below
    "bronze_directory_path": BRONZE_DIR,
    "silver_directory_path": SILVER_DIR,
  },
  "dataset": {
    "source": "directories",  # not used when h5ad already exists
    "download_params": {},
    "file_names": [],
    "overwrite": False
  },
  "harmonization": {
    "uns": {
      # Just a a dataset id that doesn't collide with others 
      "dataset_id": 2001,
      "assay": "merfish",
      "species": "mus_musculus",
      "tissue": "cortex",
    },
    "batch": {
      "split_by_batch": False,   # already split: one h5ad per section
      "batch_key": "cell_section",
      "sample_batch_map": {}     # let the harmonizer assign batch0, batch1, ...
    },
    "key_mappings": {
      # here it tells the harmonizer where to find labels/coords in .obs
      "cell_type": "cell_class",
      "X_coordinate": "coord_X",
      "Y_coordinate": "coord_Y"
    },
    "misc": {
      "min_genes": 10
    },
    "gene_mapping": {
      "keep_unmapped": True,
      "alias_map_file": "/nfs/team361/mv11/DATASETS/genes/merfish_mouse_cortex_from_luna_csv_aliases.yaml",
      "require_unique_symbols": True
    }
  }
}

cfg_dir = pl.Path(REPRO_ROOT) / "config" / "bronze_to_silver"
cfg_dir.mkdir(parents=True, exist_ok=True)
cfg_path = cfg_dir / f"{EXPERIMENT_NAME}.yaml"

with open(cfg_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Wrote config:", cfg_path)


Wrote config: /nfs/team361/mv11/vqniche-reproducibility/config/bronze_to_silver/merfish_mouse_cortex_from_luna_csv.yaml


#### harmonizer

This will read the bronze `.h5ad` files you just wrote and produce **silver** `.h5ad` files with:

* integer `X` (CSR)
* `var` containing `ensembl_id` (via `pyensembl`)
* `obsm['spatial']` as `float32` (N×2)
* harmonized metadata in `.uns` (dataset\_id, assay, species, tissue, batch)
* standardized `obs['cell_id']`, plus `obs['cell_type']` copied from `cell_class`

In [9]:
!which python


/nfs/team361/mv11/.venvs/ggsd/bin/python


In [10]:
!python {REPRO_ROOT}/analysis/data_preparation/harmonize_silver_to_bronze.py --config_file {cfg_path}


Experiment Name: merfish_mouse_cortex_from_luna_csv
Input Directory: /nfs/team361/mv11/DATASETS/bronze/merfish_mouse_cortex_from_luna_csv
Output Directory: /nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv
Using existing bronze (raw) .h5ad files...
Loading gene name to ensembl id dictionary from /nfs/team361/mv11/DATASETS/genes/mus_musculus_gene_name_to_ensembl_id_dict.pkl...
Data is already split by batch.
Harmonizing: /nfs/team361/mv11/DATASETS/bronze/merfish_mouse_cortex_from_luna_csv/merfish_mouse_cortex_mouse2_slice119.h5ad
Setting cell_type in adata.obs
Setting spatial coordinates in adata.obsm based on keys provided in key_mappings...
Raw adata shape: (5021, 254)
adata.X is present.
All entries in adata.X are integers.
  (0, 8)	3
  (0, 36)	8
  (0, 48)	3
  (0, 51)	19
  (0, 66)	3
  (0, 122)	3
  (0, 128)	3
  (0, 136)	3
  (0, 143)	3
  (0, 161)	5
  (0, 229)	13
  (0, 239)	3
  (1, 1)	2
  (1, 12)	7
  (1, 34)	5
  (1, 42)	5
  (1, 44)	2
  (1, 67)	12
  (1, 68)	32
  (1, 88

#### Quick validation of the silver outputs

In [11]:
silver_exp_dir = pl.Path(SILVER_DIR) / EXPERIMENT_NAME
silver_files = sorted(glob.glob(str(silver_exp_dir / "*.h5ad")))
print(f"Silver files: {len(silver_files)} (showing 3)\n", "\n".join(silver_files[:3]))

# Read one to sanity-check the schema VQNiche expects
ad_s = sc.read_h5ad(silver_files[0])
print(ad_s)
print("X:", type(ad_s.X), ad_s.X.dtype, ad_s.X.shape)
print("var head:\n", ad_s.var.head())
print("spatial:", ad_s.obsm['spatial'].dtype, ad_s.obsm['spatial'].shape)
print("uns:", {k: ad_s.uns[k] for k in ['dataset_id','assay','species','tissue','batch'] if k in ad_s.uns})
print("obs cols:", list(ad_s.obs.columns)[:8])


Silver files: 64 (showing 3)
 /nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv/merfish_mouse_cortex_mouse1_slice1.h5ad
/nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv/merfish_mouse_cortex_mouse1_slice10.h5ad
/nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv/merfish_mouse_cortex_mouse1_slice102.h5ad
AnnData object with n_obs × n_vars = 2360 × 254
    obs: 'coord_X', 'coord_Y', 'cell_section', 'cell_class', 'split', 'cell_type', 'n_genes', 'cell_id'
    var: 'ensembl_id'
    uns: 'assay', 'batch', 'dataset_id', 'section', 'species', 'tissue'
    obsm: 'spatial'
X: <class 'scipy.sparse._csr.csr_matrix'> int64 (2360, 254)
var head:
                        ensembl_id
gene_name                        
1700022I11Rik  ENSMUSG00000028451
1810046K07Rik                 NaN
5031425F14Rik                 NaN
5730522E02Rik  ENSMUSG00000032985
Acta2          ENSMUSG00000035783
spatial: float32 (2360, 2)
uns: {'dataset_id': 2001, 'assay': 'me

In [12]:
import scanpy as sc, pandas as pd, glob
silver_dir = "/nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv"
one = sc.read_h5ad(sorted(glob.glob(f"{silver_dir}/*.h5ad"))[0])
kept = set(one.var.index.tolist())  # the 249 kept gene symbols
all_genes = pd.read_csv("/nfs/team361/mv11/DATASETS/luna/MERFISH_mouse_cortex/MERFISH_mouse_cortex_train.csv", index_col=0).columns[:254]
dropped = [g for g in all_genes if g not in kept]
dropped


['Fam19a2', '1-Mar']

In [13]:
# QUICK CHECK — run once anywhere python can see your bronze/silver dirs
import pathlib as pl, scanpy as sc, pandas as pd

BRONZE = pl.Path("/nfs/team361/mv11/DATASETS/bronze/merfish_mouse_cortex_from_luna_csv")
SILVER = pl.Path("/nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv")

# (i) genes in bronze (from any one section, all sections have the same 254 order)
b0 = next(BRONZE.glob("*.h5ad"))
bronze_genes = pd.Index(sc.read_h5ad(b0).var_names)

# (ii) genes in silver (after harmonizer)
s0 = next(SILVER.glob("*.h5ad"))
silver_genes = pd.Index(sc.read_h5ad(s0).var_names)

missing = bronze_genes.difference(silver_genes)
print("Missing genes:", list(missing))


Missing genes: ['1-Mar', 'Fam19a2']


In [14]:
import scanpy as sc, glob
import pandas as pd

silver = sc.read_h5ad(sorted(glob.glob("/nfs/team361/mv11/DATASETS/silver/merfish_mouse_cortex_from_luna_csv/*.h5ad"))[0])
genes = pd.Index(silver.var_names)

"Fam19a2" in genes, "Tafa2" in genes, "1-Mar" in genes, "March1" in genes


(False, True, False, True)

Expected: (False, True, False, True).